|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Static batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: pick the batch size that is actually fastest<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(2)

You have measured how much faster a big batch is. Now find the batch size
that is actually fastest, which is not the same question.

All simulation, no GPU. The throughput numbers come from the demo notebook in
this folder.

In [ ]:
### run this cell

lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=20000).astype(int) + 1

# measured on an RTX 4080 Laptop in part2_pad_freeSequences. Yours will
# differ; rerun that notebook and paste your own numbers in.
speedup = {1:1.0, 2:2.0, 4:4.0, 8:8.0, 16:15.6, 32:22.7, 64:29.4, 128:32.5}
print(f'{len(lengths)} requests, median {np.median(lengths):.0f} tokens, p99 {np.percentile(lengths,99):.0f}')

# Exercise 1: how much does each batch size waste?

A static batch of B occupies B slots for `max(lengths)` steps. Useful work is
`sum(lengths)`. The rest is a finished sequence holding a slot open.

In [ ]:
def waste(batch):
  # the batch runs for max(batch) steps, in len(batch) slots.
  # how much of that rectangle is real work?
  return 

def mean_waste(B, trials=500):
  return np.mean([waste(rng.choice(lengths, B)) for _ in range(trials)])

Bs = np.array(sorted(speedup))
w  = np.array([mean_waste(int(B)) for B in Bs])
for B, x in zip(Bs, w):
  print(f'batch {B:>4}: {100*x:5.1f}% wasted')

# Exercise 2: what you offered against what you delivered

Multiply the measured speedup by the fraction of slots that did real work,
and find the peak.

In [ ]:
raw       = np.array([speedup[int(B)] for B in Bs])

# what you actually get is what the hardware offered, minus what the
# padding threw away
delivered = 

best = Bs[np.argmax(delivered)]
print(f"{'batch':>6} {'raw':>7} {'wasted':>8} {'delivered':>10}")
for B, r, x, d in zip(Bs, raw, w, delivered):
  mark = '  <-- best' if B == best else ''
  print(f'{B:>6} {r:>6.1f}x {100*x:>7.1f}% {d:>9.1f}x{mark}')

# Exercise 3: can you cheat the distribution?

The tail sets the length of the batch. So keep the tail out of the batch:
gather a larger pool, sort it by length, and cut it into batches of similar
requests.

In [ ]:
def bucketed_waste(B, n_buckets, trials=500):
  out = []
  for _ in range(trials):
    pool = rng.choice(lengths, B*n_buckets)
    # put similar lengths together, then form the batches
    
    out.append(np.mean([waste(pool[i*B:(i+1)*B]) for i in range(n_buckets)]))
  return np.mean(out)

B = int(best)
print(f'batch {B}, unsorted:            {100*mean_waste(B):5.1f}% wasted')
for nb in (2, 4, 8):
  print(f'batch {B}, sorted into {nb:>2} buckets: {100*bucketed_waste(B, nb):5.1f}% wasted')

### Before you open the solution

1. Compare the `raw` and `delivered` columns. Would raising
   `max_num_seqs` from 64 to 128 on this workload make your server
   faster or slower?
2. Bucketing by length cuts the waste sharply. What did `pool.sort()`
   need to know, and does a real server know it when the request
   arrives?
3. If you sorted by *prompt* length instead, which you do know, would
   that work? What does prompt length tell you about output length?